# Short CPU workshop: two folds, editable grid search and a window explanation

Run this notebook top to bottom. The framework performs formatting, processing, train-only normalization, grid search, training and Integrated Gradients; these cells expose the choices you need to teach and edit.

**CPU defaults:** six subjects, first two minutes each, two subject folds, five epochs per candidate and a small MLP. PaPaGei is included with a small head. This is an execution demonstration, not a benchmark; short-budget predictions can be poor.

**Before class:** download DaLiA and weights using section 2. The full archive is still about 2.7 GB even though this notebook processes a small subset. The notebook cannot make the initial download fast.

Live loss curves update in the training cells. Read the discussion prompts above each run while waiting. The participant grid block is explicitly separated from its run block; change it before running. The full `workshop.ipynb` remains the longer API reference.

In [ ]:
from pathlib import Path
import os

WORKSPACE = Path(os.environ.get('PPG_WORKSPACE', Path.cwd())).expanduser().resolve()
os.environ['PPG_WORKSPACE'] = str(WORKSPACE)
from ppg_experiment_framework.workshop import create_workspace
create_workspace(WORKSPACE, task_names=('baseline', 'exercise'))
print('Workshop workspace:', WORKSPACE)

In [ ]:
import json
from copy import deepcopy
from time import perf_counter
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Image

from ppg_experiment_framework.workshop import pipeline, plots
from ppg_experiment_framework.workshop.live import TrainingMonitor
from ppg_experiment_framework.workshop.downloads import download_file, extract_dalia

settings = pipeline.load_settings(WORKSPACE / 'configs/workshop.yaml')
profile = settings['cpu_short']
SEED = settings['seed']
DEVICE = 'cpu'
torch.set_num_threads(settings['torch_threads'])
plt.style.use('seaborn-v0_8-whitegrid')
DATA_ROOT = WORKSPACE / 'data/datasets/ppg+dalia'
ARTIFACT = 'dalia_cpu_short_2fold'
FORMATTED = DATA_ROOT / 'formatted' / ARTIFACT
PROCESSED = DATA_ROOT / 'processed' / ARTIFACT
HDF5 = DATA_ROOT / 'experiment_ready' / f'{ARTIFACT}.h5'
FIGURES = WORKSPACE / 'figures/cpu_short'
split_config = {**settings['split'], 'n_folds': 2}
RUN_PAPAGEI = True  # Set False for a minimum MLP-only session.
RUN_EXERCISE = True
print('CPU short profile:', profile)

## 1. Download once before class
Downloads are cached. PaPaGei weights are checked against the published digest. DaLiA provides wrist BVP and ECG-derived heart-rate labels. Keep the original dataset and weight notices; neither is bundled in pip.

In [ ]:
archive = download_file(settings['dalia']['url'], WORKSPACE / 'data/downloads/dalia_data.zip')
ORIGINAL = extract_dalia(archive, DATA_ROOT / 'original')
weight_settings = settings['papagei']
WEIGHTS = download_file(weight_settings['url'], WORKSPACE / 'weights/original/papagei-s/weights.pth',
                        checksum=weight_settings['checksum'], algorithm=weight_settings['algorithm'])
print('Raw subjects:', ORIGINAL)
print('Verified PaPaGei weights:', WEIGHTS)

In [ ]:
start = perf_counter()
pipeline.format_dalia(ORIGINAL, FORMATTED, subject_ids=profile['subjects'],
                      max_duration_seconds=profile['max_duration_seconds'])
metadata = json.loads((FORMATTED / 'metadata.json').read_text())
records = np.load(FORMATTED / 'dataset.npy', allow_pickle=True)
print('Formatted subjects:', len(records))

In [ ]:
example = records[0]['value']
fig, axes = plt.subplots(2, 1, figsize=(12, 5), constrained_layout=True)
visible = example['time_BVP'] < 20
axes[0].plot(example['time_BVP'][visible], example['BVP'][visible], color='#0072B2')
axes[0].set(xlabel='Time (s)', ylabel='Raw BVP (a.u.)', title='First 20 seconds of wrist BVP')
axes[1].plot(example['time_heart_rate'], example['heart_rate'], color='#009E73')
axes[1].set(xlabel='Time (s)', ylabel='Heart rate (bpm)', title='Reference labels over the selected recording')
plots.save_figure(fig, FIGURES, '01_raw_signal_and_labels')
plt.show()

## 2. Process and split by subject
We use 8-second windows at 125 Hz. Within each outer fold, two subjects train the model, one is reserved for validation and three are held out for testing. The two folds together provide one test prediction per window. Normalization learns from training subjects only.

**Discuss:** Why would randomly splitting overlapping windows produce optimistic results? What happens if normalization sees the test subjects?

In [ ]:
HDF5.parent.mkdir(parents=True, exist_ok=True)
with pipeline.make_processor(FORMATTED, PROCESSED, artifact_name=ARTIFACT) as processor:
    processor.load(overwrite_h5=True)
    processor.resample(labels=['BVP'], target_fs=settings['sample_rate'],
                       method='interpolation', timestamps_mode='regenerate', processor_backend='sequential')
    processor.window(biosignal_labels=['BVP'], target_labels=['heart_rate'],
                     window_time_seconds=settings['window_seconds'], target_time_label='time_heart_rate',
                     signal_time_label='time_BVP', processor_backend='sequential')
    processor.impute(labels=['heart_rate'], method='complete_case', processor_backend='sequential')
    dataset = processor.make_experiment_ready(x_train_labels=['BVP'], y_train_labels=['heart_rate'],
                    hdf5_path=HDF5, group_by='subject_id', overwrite=True,
                    verify_integrity=True, export_batch_size=1024)
print(dataset)

In [ ]:
split_path, transform_path = pipeline.save_training_splits(dataset, split_config,
    directory=HDF5.with_suffix('') / 'splits', val_fraction=settings['val_fraction'])
split_table = pipeline.split_summary(split_path)
display(split_table)
fig = plots.plot_splits(split_table)
plots.save_figure(fig, FIGURES, '02_subject_splits')
plt.show()
fig = plots.plot_waveforms(dataset, indices=[0], sample_rate=settings['sample_rate'])
plots.save_figure(fig, FIGURES, '03_window_and_derivatives')
plt.show()
print('Ready:', dataset.n_samples, 'windows. Processing seconds:', round(perf_counter() - start, 1))

## 3. Instructor example: two-candidate grid search
The framework tries both learning rates inside each outer fold and chooses the candidate with the lowest validation MSE for that fold. It tests only the selected candidate on that fold's held-out subjects. This uses a single subject holdout for validation, not an inner cross-validation loop.

**While it runs:** Which curve would suggest overfitting? Why can the selected learning rate differ across folds? Why must the test score stay out of the selection rule?

The live display shows training/validation loss for the current fit and validation trajectories for all candidates seen so far. MSE is in bpm squared. Each fit starts from the same seeded initialization.

In [ ]:
mlp_config = pipeline.load_settings(WORKSPACE / 'configs/mlp.yaml')
mlp_config['backbone']['params'].update(signal_length=int(settings['sample_rate'] * settings['window_seconds']), hidden_size=64)
experiment_config = pipeline.load_settings(WORKSPACE / 'configs/experiment.yaml')
experiment_config['training'].update(n_epochs=profile['epochs'], device=DEVICE, batch_size=32,
                                      log_every_epochs=1, progress_bar=False)
experiment_config['testing']['device'] = DEVICE
experiment_config['split_configs'] = [split_config]
experiment_config['training']['early_stopping']['patience'] = profile['epochs']
experiment_config['training']['grid_search_config']['param_grid'] = {'learning_rate': [0.003, 0.01]}
display(pipeline.describe_model(mlp_config))
print('2 candidates x 2 folds = 4 fits; at most', 4 * profile['epochs'], 'epochs total.')
with TrainingMonitor(title='Instructor MLP grid: live training') as baseline_monitor:
    mlp_run = pipeline.run_task(dataset=dataset, base_config=mlp_config, experiment_config=experiment_config,
        split_path=split_path, transform_path=transform_path, output_dir='results/cpu_short_mlp_grid', seed=SEED)
mlp_predictions = pipeline.prediction_table(mlp_run, model_name='MLP')
validation_table = pipeline.grid_results(mlp_run)
display(validation_table)
assert len(validation_table) == 4 and validation_table.selected.sum() == 2
fig = baseline_monitor.figure()
plots.save_figure(fig, FIGURES, '04_learning_curves')
plt.show()
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
validation_table.pivot(index='learning_rate', columns='fold', values='validation_MSE').plot.bar(ax=ax)
ax.set(xlabel='Learning rate', ylabel='Best validation MSE', title='Choose settings using validation subjects')
plots.save_figure(fig, FIGURES, '05_grid_validation')
plt.show()

## 4. Optional foundation-model comparison
PaPaGei-S stays frozen and the framework trains a small head. The same subject splits and training-only input normalization apply. This uses one candidate to keep the CPU workload small.

**While it runs:** Which parameters change? What does precomputing frozen embeddings save? Why is this small subset insufficient for a benchmark claim?

In [ ]:
predictions = mlp_predictions.copy()
if RUN_PAPAGEI:
    papagei_config = pipeline.load_settings(WORKSPACE / 'configs/papagei_s.yaml')
    head_config = pipeline.load_settings(WORKSPACE / 'configs/head.yaml')
    head_config['head']['params']['mlp_config']['hidden_dims'] = [64]
    papagei_experiment = deepcopy(experiment_config)
    papagei_experiment['training']['grid_search_config']['param_grid'] = {'learning_rate': [0.01]}
    with TrainingMonitor(title='Frozen PaPaGei + head: live training') as papagei_monitor:
        papagei_run = pipeline.run_task(dataset=dataset, base_config=papagei_config, head_config=head_config,
            experiment_config=papagei_experiment, split_path=split_path, transform_path=transform_path,
            output_dir='results/cpu_short_papagei', seed=SEED)
    predictions = pd.concat([predictions, pipeline.prediction_table(papagei_run, model_name='PaPaGei-S')], ignore_index=True)
display(pipeline.regression_metrics(predictions))

## 5. Inspect predictions and errors
Positive signed error means prediction minus reference. Look at agreement, model error distributions and errors against the mean absolute first/second derivatives of the raw processed signal. These are descriptive associations, not causal explanations.

In [ ]:
fig = plots.plot_agreement(predictions)
plots.save_figure(fig, FIGURES, '05_prediction_agreement')
plt.show()
fig = plots.plot_error_relationships(predictions, colors=settings['plot_colors'])
plots.save_figure(fig, FIGURES, '06_error_vs_prediction_and_reference')
plt.show()

In [ ]:
fig = plots.plot_error_violins(predictions)
plots.save_figure(fig, FIGURES, '07_error_violins')
plt.show()
fig = plots.plot_error_violins(predictions, by_subject=True)
plots.save_figure(fig, FIGURES, '08_subject_error_violins')
plt.show()

In [ ]:
features = pipeline.signal_features(dataset, sample_rate=settings['sample_rate'])
diagnostics = pipeline.join_signal_features(predictions, features)
display(diagnostics.head())
derivative_columns = ('mean_abs_first_derivative', 'mean_abs_second_derivative')
derivative_labels = ('Mean |dBVP/dt| (a.u./s)', 'Mean |d²BVP/dt²| (a.u./s²)')
fig = plots.plot_error_relationships(diagnostics, x_columns=derivative_columns,
                    x_labels=derivative_labels, colors=settings['plot_colors'])
plots.save_figure(fig, FIGURES, '10_signed_error_vs_derivatives')
plt.show()
fig = plots.plot_error_relationships(diagnostics, x_columns=derivative_columns,
                    x_labels=derivative_labels, absolute=True, colors=settings['plot_colors'])
plots.save_figure(fig, FIGURES, '11_absolute_error_vs_derivatives')
plt.show()

## 6. Your turn: edit the grid below
Change the **learning-rate list** and optionally try **two batch sizes**. Make your hypothesis before running. All combinations are tried by the framework, with validation selecting the winner separately in each fold.

Keep the epoch count, model and saved splits fixed for this exercise. Start with two candidates; expanding to two learning rates and two batch sizes produces four candidates and eight fits. The safety check below caps the classroom grid at four candidates; deliberately change the cap only if you have more time.

A model parameter such as hidden size is a separate model-config change; this block searches supported training parameters. Compare validation results first. Repeated test-driven tuning invalidates final evaluation.

In [ ]:
# EDIT THIS BLOCK, then run the separate training block below.
PARTICIPANT_GRID = {
    'learning_rate': [0.001, 0.02],
    'batch_size': [32],  # Try [16, 32] for a four-candidate grid.
}
MAX_CANDIDATES = 4
candidate_count = int(np.prod([len(values) for values in PARTICIPANT_GRID.values()]))
assert 1 <= candidate_count <= MAX_CANDIDATES, 'Keep this classroom search small.'
print(candidate_count, 'candidates x 2 folds =', candidate_count * 2, 'fits')

In [ ]:
participant_experiment = deepcopy(experiment_config)
participant_experiment['training']['grid_search_config']['param_grid'] = PARTICIPANT_GRID
comparison = predictions.copy()
if RUN_EXERCISE:
    with TrainingMonitor(title='Your editable grid: live training') as participant_monitor:
        participant_run = pipeline.run_task(dataset=dataset, base_config=mlp_config,
            experiment_config=participant_experiment, split_path=split_path, transform_path=transform_path,
            output_dir='results/cpu_short_participant_grid', seed=SEED)
    participant_validation = pipeline.grid_results(participant_run)
    display(participant_validation)
    comparison = pd.concat([predictions, pipeline.prediction_table(participant_run, model_name='MLP exercise')], ignore_index=True)
    assert participant_validation.selected.sum() == 2
    fig = participant_monitor.figure()
    plots.save_figure(fig, FIGURES, '12_participant_learning_curves')
    plt.show()
display(pipeline.regression_metrics(comparison))
fig = plots.plot_error_violins(comparison)
plots.save_figure(fig, FIGURES, '13_participant_error_comparison')
plt.show()

## 7. Afterwards: Integrated Gradients on one held-out window
We explain the **instructor MLP's validation-selected checkpoint**, using the framework's saved split normalization. The framework verifies reconstructed predictions against the original predictions before attribution.

To keep CPU time small, export every fourth source window and inspect the first exported window of fold 0. Selection is fixed by index, not by how convincing its explanation looks. Set `WINDOW_POSITION` below to inspect another exported window.

A zero baseline is used in normalized input space. Red/blue indicate positive/negative contributions relative to that baseline. The convergence delta measures the IG completeness approximation; it is not an accuracy score. Attributions describe model sensitivity, not physiological causality.

In [ ]:
import h5py
from ppg_experiment_framework.XAI.run_integrated_gradients import export_run_integrated_gradients
from ppg_experiment_framework.XAI.visualize_ppg_dalia_ig import visualize_ppg_dalia_ig

ig_path = export_run_integrated_gradients(mlp_run['run_dir'], split_selection=['0'], device='cpu',
    n_steps=32, batch_size=8, integration_batch_size=8, source_entry_stride=4, overwrite=True)
WINDOW_POSITION = 0  # Edit after the first run to inspect another exported window.
with h5py.File(ig_path, 'r') as handle:
    group = handle['splits/split_0']
    assert 0 <= WINDOW_POSITION < len(group['sample_index'])
    chosen_index = int(group['sample_index'][WINDOW_POSITION])
    display(pd.Series({'sample_index': chosen_index, 'subject': group['group_id'].asstr()[WINDOW_POSITION],
        'reference_bpm': float(group['target'][WINDOW_POSITION, 0]),
        'prediction_bpm': float(group['prediction'][WINDOW_POSITION, 0]),
        'convergence_delta': float(group['convergence_delta'][WINDOW_POSITION, 0])}))
ig_summary = visualize_ppg_dalia_ig([('MLP', ig_path)], output_dir=FIGURES / 'integrated_gradients',
    sampling_rate=settings['sample_rate'], sample_index=chosen_index, max_heatmap_windows=32)
display(Image(filename=ig_summary['figures']['representative_window']))
print('Additional population and convergence figures:', FIGURES / 'integrated_gradients')

## 8. Keep the evidence
Save settings, validation tables and held-out metrics. Outputs from the short notebook live under `results/cpu_short_*` (model runs), `results/cpu_short/summary` (tables) and `figures/cpu_short` so the longer tour remains separate. Read `workshop.ipynb` afterwards for streaming, custom transforms, window sequences and other framework APIs.

In [ ]:
OUTPUT = WORKSPACE / 'results/cpu_short/summary'
OUTPUT.mkdir(parents=True, exist_ok=True)
comparison.to_csv(OUTPUT / 'held_out_predictions.csv', index=False)
pipeline.regression_metrics(comparison).to_csv(OUTPUT / 'pooled_metrics.csv', index=False)
pipeline.regression_metrics(comparison, by=('model', 'subject')).to_csv(OUTPUT / 'subject_metrics.csv', index=False)
validation_table.to_csv(OUTPUT / 'instructor_validation.csv', index=False)
if RUN_EXERCISE:
    participant_validation.to_csv(OUTPUT / 'participant_validation.csv', index=False)
pipeline.save_run_manifest(OUTPUT / 'manifest.json', {'profile': profile, 'split_config': split_config,
    'experiment_config': experiment_config, 'participant_grid': PARTICIPANT_GRID, 'mlp_config': mlp_config})
print('Short CPU workshop complete. Results:', OUTPUT)